# Lab - Chat Server: Sessions and a JSON Contract

Three tasks on the server between a chat page and Claude - the integration boundary. The page sends JSON, the server decides what it accepts, which history a message belongs to, and what goes back.

| Task | What you learn |
| --- | --- |
| 1 | The /chat endpoint and its contract: session_id and text in, reply out |
| 2 | Two browser tabs, two sessions - the session id picks the history |
| 3 | A malformed request gets 422 and never reaches Claude |

**You do not need an API key, a web server or a browser.** This lab ships with a Claude simulator, and stand-ins for FastAPI and pydantic when they are not installed. `browser_post()` plays the chat page. The server code is exactly the code you would run with uvicorn at home.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check, browser_post   # browser_post plays the chat page

# Everything below this line is ordinary server code.
from dotenv import load_dotenv
from anthropic import Anthropic
from fastapi import FastAPI
from pydantic import BaseModel

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Answer in one or two sentences."
)
HISTORIES = {}   # session id -> that chat's history, kept on the server

# The lecture's script ends with one more line that serves the chat page itself:
#     app.mount("/", StaticFiles(directory="static", html=True))
# This lab has no browser - browser_post() plays the page - so that line is left out.

---

## Task 1 - The endpoint and its JSON contract

In [ ]:
# ============================================================
# TASK 1 - The endpoint and its JSON contract
# ============================================================
#
# WHAT TO DO
#   Describe what the page sends - a session id and a piece of
#   text - and write the /chat endpoint. It keeps one history
#   per session, sends that history with the system line to
#   Claude, stores the reply, and returns JSON.
#
# WHY IT MATTERS
#   The page and the server agree on a contract: what goes in
#   (ChatIn) and what comes out (reply and input_tokens). The
#   lecture's server keeps one history for everyone; a real one
#   serves many chats at once, so the page sends its session id
#   and the server keeps a history per session.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the one thing this server does".
#
# HOW TO DO IT
#   Three blanks:
#
#       str               the type of the session id
#       body.session_id   which history this message belongs to
#       "/chat"           the address the page sends to
# ============================================================

app = FastAPI()   # provided: a fresh app each time you run this cell

# BEGIN SOLUTION
class ChatIn(BaseModel):
    session_id: str
    text: str


@app.post("/chat")
def chat(body: ChatIn):
    history = HISTORIES.setdefault(body.session_id, [])
    history.append({"role": "user", "content": body.text})
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        output_config={"effort": "low"},
        system=SYSTEM,
        messages=history,
    )
    for block in response.content:
        if block.type == "text":
            reply = block.text
    history.append({"role": "assistant", "content": reply})
    return {"reply": reply, "input_tokens": response.usage.input_tokens}
# SCAFFOLD: class ChatIn(BaseModel):
# SCAFFOLD:     session_id: ...   # str
# SCAFFOLD:     text: str
# SCAFFOLD:
# SCAFFOLD:
# SCAFFOLD: @app.post(...)   # "/chat"
# SCAFFOLD: def chat(body: ChatIn):
# SCAFFOLD:     history = HISTORIES.setdefault(..., [])   # body.session_id
# SCAFFOLD:     history.append({"role": "user", "content": body.text})
# SCAFFOLD:     response = client.messages.create(
# SCAFFOLD:         model=MODEL,
# SCAFFOLD:         max_tokens=200,
# SCAFFOLD:         output_config={"effort": "low"},
# SCAFFOLD:         system=SYSTEM,
# SCAFFOLD:         messages=history,
# SCAFFOLD:     )
# SCAFFOLD:     for block in response.content:
# SCAFFOLD:         if block.type == "text":
# SCAFFOLD:             reply = block.text
# SCAFFOLD:     history.append({"role": "assistant", "content": reply})
# SCAFFOLD:     return {"reply": reply, "input_tokens": response.usage.input_tokens}
# END SOLUTION: replace each ... below with the value named beside it

HISTORIES.clear()   # provided: a fresh server
page = browser_post(app, "/chat", {"session_id": "tab-1",
                                   "text": "I bought a jacket twenty days ago."})

check("chat_contract", app=app, histories=HISTORIES, page=page)

---

## Task 2 - Two browser tabs, two sessions

In [ ]:
# ============================================================
# TASK 2 - Two browser tabs, two sessions
# ============================================================
#
# WHAT TO DO
#   Two customers have the chat open in two tabs. Tab 1
#   continues its conversation; tab 2 asks the same question
#   with its own session id. Compare the replies and the input
#   tokens.
#
# WHY IT MATTERS
#   The page sends only the new text - the server kept the
#   history, so Claude knows about the jacket, and the input
#   tokens grow just like in the script. But the history it
#   uses is chosen by the session id: tab 2 must never get an
#   answer built on tab 1's conversation.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The server kept the history, so Claude knows about the
#   jacket".
#
# HOW TO DO IT
#   Three blanks:
#
#       "tab-1"                    tab 1's session id
#       "tab-2"                    tab 2's own session id
#       "Can I still return it?"   tab 2's question
# ============================================================

HISTORIES.clear()   # provided: a fresh server
try:
    # BEGIN SOLUTION
    pages = {
        "tab-1": [
            browser_post(app, "/chat", {"session_id": "tab-1",
                                        "text": "I bought a jacket twenty days ago."}),
            browser_post(app, "/chat", {"session_id": "tab-1",
                                        "text": "Can I still return it?"}),
        ],
        "tab-2": [
            browser_post(app, "/chat", {"session_id": "tab-2",
                                        "text": "Can I still return it?"}),
        ],
    }
    # SCAFFOLD: pages = {
    # SCAFFOLD:     "tab-1": [
    # SCAFFOLD:         browser_post(app, "/chat", {"session_id": "tab-1",
    # SCAFFOLD:                                     "text": "I bought a jacket twenty days ago."}),
    # SCAFFOLD:         browser_post(app, "/chat", {"session_id": ...,   # "tab-1"
    # SCAFFOLD:                                     "text": "Can I still return it?"}),
    # SCAFFOLD:     ],
    # SCAFFOLD:     "tab-2": [
    # SCAFFOLD:         browser_post(app, "/chat", {"session_id": ...,   # "tab-2"
    # SCAFFOLD:                                     "text": ...}),       # "Can I still return it?"
    # SCAFFOLD:     ],
    # SCAFFOLD: }
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for tab, replies in pages.items():
    print(tab, "input tokens:", [p["input_tokens"] for p in replies])

check("tab_sessions", pages=pages, histories=HISTORIES)

---

## Task 3 - A malformed request never reaches Claude

In [ ]:
# ============================================================
# TASK 3 - A malformed request never reaches Claude
# ============================================================
#
# WHAT TO DO
#   Play a broken page that forgets the text field, and look at
#   what the server answers - and at what it did not do.
#
# WHY IT MATTERS
#   The browser is the customer's, and anything can arrive from
#   it. The server keeps the key, the rules and the
#   conversation on our side - and it is also where input gets
#   checked. A body that does not fit ChatIn is answered with
#   422 before the endpoint runs: no history entry, no request
#   to Claude, no tokens spent.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The server keeps the key, the rules and the conversation
#   on our side".
#
# HOW TO DO IT
#   Three blanks:
#
#       "tab-3"          the broken page's session id
#       "/chat"          the same endpoint
#       "status"         the field that holds the HTTP status
# ============================================================

try:
    # BEGIN SOLUTION
    rejected = browser_post(app, "/chat", {"session_id": "tab-3"})   # no "text"
    print("status:", rejected["status"])
    # SCAFFOLD: rejected = browser_post(app, ..., {"session_id": ...})   # "/chat", "tab-3" - no "text"
    # SCAFFOLD: print("status:", rejected[...])   # "status"
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError, KeyError, ValueError, LookupError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("sessions on the server:", sorted(HISTORIES))

check("bad_request", rejected=rejected, histories=HISTORIES)

---

## Done

The browser talks to your server, and only your server talks to Claude. The JSON contract says what may come in and what goes out, the session id keeps chats apart, and a request that breaks the contract stops at the boundary.